# Benchmark techniczny modeli detekcji obiektów dla zadania pomiaru ruchu drogowego
Celem notebooka jest porównanie wybranych modeli detekcji obiektów pod kątem ich przydatności jako pierwszego etapu systemu pomiaru ruchu drogowego z nagrań wideo. Na tym etapie oceniane są: poprawność techniczna uruchomienia modelu, czas inferencji, liczba wykrytych obiektów należących do klas ruchu drogowego oraz możliwość ujednolicenia wyników do wspólnego formatu.

Notebook nie ocenia jeszcze pełnej dokładności pomiaru ruchu drogowego, ponieważ nie obejmuje trackingu, zliczania przekroczeń linii pomiarowej ani porównania z anotacjami referencyjnymi.

---

### Zestawienie modeli detekcji wziętych pod uwagę

| Nazwa modelu | Architektura modelu | Właściciel praw | Licencja | Dataset treningowy | AP (wg. oficjalnego źródła) |
| -- | -- | -- | -- | -- | -- |
| fasterrcnn_resnet50_fpn | CNN (ResNet-50 + FPN, Faster R-CNN) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 37.0 |
| fasterrcnn_mobilenet_v3_large_fpn | CNN (MobileNetV3-Large + FPN, Faster R-CNN) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 32.8 |
| retinanet_resnet50_fpn | CNN (ResNet-50 + FPN, RetinaNet) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 36.4 |
| retinanet_resnet50_fpn_v2 | CNN (ResNet-50 + FPN, RetinaNet v2) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 41.5 |
| fcos_resnet50_fpn | CNN (ResNet-50 + FPN, anchor-free FCOS) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 39.2 |
| ssd300_vgg16 | CNN (VGG16, SSD300) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 25.1 |
| ssdlite320_mobilenet_v3_large | CNN (MobileNetV3-Large, SSDLite320) | PyTorch / Torchvision | BSD-3-Clause | COCO train2017 | 21.3 |
| facebook/detr-resnet-50 | Hybrydowa (CNN backbone + Transformer) | Facebook / Meta AI| Apache-2.0 | COCO 2017 | 42.0 |
| microsoft/conditional-detr-resnet-50 | Hybrydowa (CNN backbone + Transformer) | Microsoft | Apache-2.0 | COCO 2017 | — |
| SenseTime/deformable-detr | Hybrydowa (CNN backbone + Transformer) | SenseTime | Apache-2.0 | COCO 2017 | — |
| PekingU/rtdetr_r18vd | Hybrydowa (CNN backbone + Transformer) | Peking University (PekingU) | Apache-2.0 | COCO train2017 | 46.5 |

### Przygotowanie środowiska

In [1]:
# Import all neccessary libraries
import os
import sys
import time
import psutil
import torch
import numpy as np
import PIL.Image as Image
from pathlib import Path
from huggingface_hub.utils import disable_progress_bars
from transformers.utils import logging as transformers_logging

# silence huggingface logging
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY_LEVEL"] = "error"
disable_progress_bars()

# silence transformers logging
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_VERBOSITY_LEVEL"] = "error"
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()


# import pyvision models
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, fasterrcnn_mobilenet_v3_large_fpn,
    retinanet_resnet50_fpn, retinanet_resnet50_fpn_v2,
    fcos_resnet50_fpn, ssd300_vgg16, ssdlite320_mobilenet_v3_large
)
from torchvision.models import list_models, get_model, get_model_weights
# import transformers utils
from transformers import AutoModelForObjectDetection, AutoImageProcessor

# mount google drive test data
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

MAPILLARY_TEST_IMAGE = Path('/content/drive/MyDrive/colab/mapilary_test_data/')
REAL_LIFE_TEST_VIDEO = Path('/content/drive/MyDrive/colab/test_data/irl_video/')


In [2]:
# define constant model list
MODELS = [
    "fasterrcnn_resnet50_fpn",
    "fasterrcnn_mobilenet_v3_large_fpn",
    "retinanet_resnet50_fpn",
    "retinanet_resnet50_fpn_v2",
    "fcos_resnet50_fpn",
    "ssd300_vgg16",
    "ssdlite320_mobilenet_v3_large",
    "facebook/detr-resnet-50",
    "microsoft/conditional-detr-resnet-50",
    "SenseTime/deformable-detr",
    "PekingU/rtdetr_r18vd"
]

TV_MODELS = set(list_models())
# define test environment

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"DEBUG: Using device: {device}")

In [3]:
# loading model function
def load_model(model_name, device):
    if model_name in TV_MODELS:
        # print(f"DEBUG: Loading torchvision model: {model_name}")
        weights = get_model_weights(model_name).DEFAULT
        model = get_model(model_name, weights=weights).to(device).eval()
        preprocessing = weights.transforms()

        meta = {
            "model_name": model_name,
            "type": "torchvision",
            "weights": str(weights),
            "categories": weights.meta.get("categories"),
        }
        return model, preprocessing, meta
    
    # print(f"DEBUG: Loading transformers model: {model_name}")
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModelForObjectDetection.from_pretrained(model_name).to(device).eval()
    meta = {
        "model_name": model_name,
        "type": "transformers",
        "weights": "from pretrained",
        "categories": model.config.id2label,
    }
    return model, processor, meta

# look for meta parameters in the model metadata
# this is to check if the metadata contains the necessary information for benchmarking

def check_meta_parameters(model, model_name):
    meta_params = [
        name for name, param in model.named_parameters()
        if getattr(param, "is_meta", False)
    ]

    meta_buffers = [
        name for name, buffer in model.named_buffers()
        if getattr(buffer, "is_meta", False)
    ]

    if meta_params or meta_buffers:
        print(f"[WARNING] {model_name}: model still has meta tensors")
        print(f"Meta parameters: {len(meta_params)}")
        print(f"Meta buffers: {len(meta_buffers)}")
        print("First examples:", meta_params[:10])
        return False

    print(f"[OK] {model_name}: no meta tensors detected")
    return True 

In [4]:
# loding model and logging metadata

loaded_models = {}

for n, model_name in enumerate(MODELS, start=1):
    try:
        model, processor, metadata = load_model(model_name, device)
        print(f"[{n}/{len(MODELS)}] Loaded {model_name} \nwith metadata: {metadata}")
        check_meta_parameters(model, model_name)
        loaded_models[model_name] = (model, processor, metadata)
    except Exception as e:
        print(f"ERROR while loading {model_name}: {e}")

### Logika detekcji

In [16]:
# define objects of interest for benchmarking and select their corresponding class ids from the model metadata

OBJECTS = [
    "person",
    "pedestrian",
    "bicycle",
    "bike",
    "car",
    "bus",
    "truck",
    "motorcycle",
    "motorbike",
]

def get_object_id(metadata, objects=OBJECTS):
    model_name = metadata["model_name"]
    categories = metadata["categories"]
    selected_ids = {}
    if isinstance(categories, list):
        iterable = enumerate(categories)
    elif isinstance(categories, dict):
        iterable = categories.items()
    else:
        print(f"[ERROR] {model_name}: unknown categories format: {type(categories)}")
    
    for class_id, class_name in iterable:
        class_id = int(class_id)
        class_name = class_name.lower().strip()
        if class_name in objects:
            selected_ids[class_id] = class_name
    return selected_ids

In [ ]:
# define method for objrect detection on the still image

def detect_from_image(model_name, model_bundle, device, image_path, objects=OBJECTS):
    curr_model, processor, metadata = model_bundle
    print(f"DEBUG: Model name {model_name}")

    object_ids = get_object_id(metadata)
    
    image = Image.open(image_path).convert("RGB")
    processed_image = processor(image).to(device)

    if model_name in TV_MODELS:
        with torch.inference_mode():
            prediction = curr_model([processed_image])[0]
            detections = []

        for box, label_id, score in zip(
            prediction["boxes"],
            prediction["labels"],
            prediction["scores"],
        ):
            label_id = int(label_id)
            score = float(score)

            if label_id not in object_ids:
                continue
            if score < 0.5:
                continue

            detections.append({
                "class_id": label_id,
                "class_name": metadata["categories"][label_id],
                "score": score,
                "box": box.detach().cpu().tolist(),
            })
    else:
        detections=[]
        with torch.no_grad():
            outputs = curr_model(**inputs)
        target_sizes = torch.tensor([image.size[::-1]], device=device)
        results = processor.post_process_object_detection(
            outputs,
            treshold=0.5,
            target_sizes=target_sizes
        )
        for box, label_id, score in zip(
            results["boxes"],
            results["labels"],
            results["scores"]
        ):
            if label_id not in object_ids:
                continue

            detections.append({
                "class_id": label_id,
                "class_name": metadata["categories"][label_id],
                "score": score,
                "box": box.detach().cpu().tolist(),
            })

    return detections

def detection_runner():
    outputs = {}
    image_dir = MAPILLARY_TEST_IMAGE / "images"
    for model_name, model_bundle in loaded_models.items():
        outputs[model_name] = {}
        for image_name in os.listdir(image_dir):
            print(f"{model_name}: {image_name}")
            detections = detect_from_image(
                model_name=model_name,
                model_bundle=model_bundle,
                device=device,
                image_path=image_dir / image_name,
            )
            outputs[model_name][image_name] = detections

detection_runner()

In [14]:
for model_name, image_outputs in outputs.items():
    print(f"\n{model_name}")
    for image_name, detections in image_outputs.items():
        det_count = len(detections) if isinstance(detections, list) else 0
        print(f"  {image_name}: {det_count} detections")

## Opracowanie danych kontrolnych

In [15]:
import json


CONFIG_PATH = MAPILLARY_TEST_IMAGE / "config.json"

with open(CONFIG_PATH, "r") as f:
    mapillary_config = json.load(f)


MAPILLARY_TARGET_NAMES = {
    "human--person": "pedestrian",
    "human--rider--bicyclist": "bicyclist",
    "human--rider--motorcyclist": "motorcyclist",
    "human--rider--other-rider": "rider",

    "object--vehicle--bicycle": "bicycle",
    "object--vehicle--bus": "vehicle",
    "object--vehicle--car": "vehicle",
    "object--vehicle--caravan": "vehicle",
    "object--vehicle--motorcycle": "vehicle",
    "object--vehicle--other-vehicle": "vehicle",
    "object--vehicle--trailer": "vehicle",
    "object--vehicle--truck": "vehicle",
    "object--vehicle--wheeled-slow": "vehicle",
}


def build_mapillary_target_ids(config, target_names):
    target_ids = {}

    for class_id, label in enumerate(config["labels"]):
        label_name = label["name"]

        if label_name in target_names:
            target_ids[class_id] = {
                "readable": label["readable"],
                "name": label["name"],
                "target_class": target_names[label_name],
                "instances": label["instances"],
                "evaluate": label["evaluate"],
            }

    return target_ids


MAPILLARY_TARGET_IDS = build_mapillary_target_ids(
    mapillary_config,
    MAPILLARY_TARGET_NAMES
)

MAPILLARY_TARGET_IDS